In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [2]:
CHEXPERT_PATH = "/kaggle/input/datasets/ashery/chexpert"
MODEL_PATH = "/kaggle/input/models/ethanmails/efficient-net-b1/pytorch/default/1/efficient_net-b1_model.pth"

In [3]:
!ls /kaggle/input/datasets/ashery/chexpert

train  train.csv  valid  valid.csv


In [4]:
train_csv = pd.read_csv(f"{CHEXPERT_PATH}/train.csv")
valid_csv = pd.read_csv(f"{CHEXPERT_PATH}/valid.csv")

In [5]:
TARGET_COLS = [
    'Atelectasis',
    'Cardiomegaly',
    'Consolidation',
    'Edema',
    'Pleural Effusion',
    'Pneumonia',
    'Pneumothorax'
]

In [6]:
for col in TARGET_COLS:
    train_csv[col] = train_csv[col].fillna(0)
    valid_csv[col] = valid_csv[col].fillna(0)

    train_csv[col] = train_csv[col].replace(-1, 0)
    valid_csv[col] = valid_csv[col].replace(-1, 0)

In [7]:
class CheXpertDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = '/kaggle/input/datasets/ashery/chexpert/' + row['Path'].replace('CheXpert-v1.0-small/', '')
        
        image = cv2.imread(img_path)
        
        if image is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        labels = row[TARGET_COLS].values.astype(np.float32)
        
        if self.transforms:
            image = self.transforms(image=image)['image']
        
        return image, torch.tensor(labels)

In [8]:
train_transforms = A.Compose([
    A.Resize(384, 384),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Resize(384, 384),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

In [9]:
train_dataset = CheXpertDataset(train_csv,transforms=train_transforms)
valid_dataset = CheXpertDataset(valid_csv,transforms=valid_transforms)

In [10]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True,num_workers=4,pin_memory=True)
valid_loader = DataLoader(valid_dataset,batch_size=64,shuffle=False,num_workers=4,pin_memory=True)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [12]:
import torchvision.models as models
import torch.nn as nn

class EfficientNetMultiLabel(nn.Module):
    def __init__(self, model_name='efficientnet_b1', num_classes=14, pretrained=True):
        super().__init__()

        if pretrained:
            self.backbone = models.efficientnet_b1(weights='DEFAULT')
        else:
            self.backbone = models.efficientnet_b1(weights=None)

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

In [13]:
model = EfficientNetMultiLabel(num_classes=14,pretrained=False).to(device)

In [14]:
checkpoint = torch.load(MODEL_PATH,map_location=device,weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [15]:
num_chexpert_classes = len(TARGET_COLS)

in_features = model.backbone.classifier[1].in_features

model.backbone.classifier = nn.Sequential(nn.Dropout(0.4),nn.Linear(in_features, num_chexpert_classes))

In [16]:
for param in model.backbone.features.parameters():
    param.requires_grad = False

In [17]:
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)

In [18]:
@torch.no_grad()
def validate(model, loader):
    model.eval()

    preds_all = []
    labels_all = []

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = torch.sigmoid(outputs)

        preds_all.append(preds.cpu().numpy())
        labels_all.append(labels.cpu().numpy())

    preds_all = np.concatenate(preds_all)
    labels_all = np.concatenate(labels_all)

    auc_scores = []

    for i in range(len(TARGET_COLS)):
        try:
            auc = roc_auc_score(labels_all[:, i], preds_all[:, i])
            auc_scores.append(auc)
        except:
            pass

    return np.mean(auc_scores)

In [19]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0

    loop = tqdm(train_loader)

    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    val_auc = validate(model, valid_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {running_loss / len(train_loader):.4f}")
    print(f"Validation AUROC: {val_auc:.4f}")

100%|██████████| 4/4 [00:03<00:00,  1.08it/s]


Epoch 1
Train Loss: 0.3643
Validation AUROC: 0.7767


100%|██████████| 4/4 [00:02<00:00,  1.72it/s]


Epoch 2
Train Loss: 0.3410
Validation AUROC: 0.7948


100%|██████████| 4/4 [00:02<00:00,  1.59it/s]


Epoch 3
Train Loss: 0.3386
Validation AUROC: 0.8007


100%|██████████| 4/4 [00:02<00:00,  1.75it/s]


Epoch 4
Train Loss: 0.3371
Validation AUROC: 0.8082


100%|██████████| 4/4 [00:02<00:00,  1.80it/s]

Epoch 5
Train Loss: 0.3366
Validation AUROC: 0.8050


In [20]:
for param in model.parameters():
    param.requires_grad = True

In [21]:
optimizer = optim.AdamW(model.parameters(),lr=1e-5,weight_decay=1e-5)

In [22]:
torch.save(model.state_dict(),'./efficient_net-b1_fine-tuned.pth')